# Drive a Scenario

The driver's lifecycle: bundle a rolled world, a route and a driver into a scenario file, drive it in CARLA, then check the recorded telemetry against the safety requirements.

The driver follows whatever route it is given. It reads the hazards that actually occur, never the forecast's probabilities.

In [ ]:
from pathlib import Path

from pyotter import Driver, Planner, Visualize, World
from pyotter.core import DrivingSafetyReq, WorldHazardLibrary, WorldRoadGraph
from pyotter.modules.driver import DriverModel, DrivingScenarioTelemetry
from pyotter.modules.planner import PlannerConfig

OUTPUT = Path("runs")
OUTPUT.mkdir(exist_ok=True)

## The World, One Roll of It, and Three Routes

The forecast is the one from `world_demo.ipynb`, and the trip is the one from `planner_demo.ipynb`. One seed rolls the hazards, so all three drivers meet the same world.

In [ ]:
hazards = WorldHazardLibrary()
wrg = WorldRoadGraph.from_template("town10")
for lane in wrg.segments_in_junction(189):
    lane.assign_hazard_probability([(hazards.pedestrian, 0.03)])
for lane in wrg.segments_in_junction(841):
    lane.assign_hazard_probability([(hazards.pedestrian, 0.01)])
wrg.set_condition(hazards.wet_road, probability=0.9, severity="high", variability=0.1)
wrg.set_condition(hazards.low_visibility, probability=0.5, severity="mid", variability=0.4)
wrg.set_time_of_day(17.5)

SEED = 1227
spawns = World.roll_hazards(wrg, hazards, seed=SEED, duration_s=150)

A, B = wrg.spawn_points[18], wrg.spawn_points[127]
config = PlannerConfig(weights={"pedestrian": 20.0})
routes = {mode: Planner.route(A, B, wrg, mode=mode, risk_scale=2, config=config)
          for mode in ("naive", "cautious", "risk-aware")}

for condition in spawns.conditions:
    print(f"{condition.hazard_id}: severity {condition.severity:.2f} ({condition.level})")
print(f"sun: facing it loses {spawns.sun.strength:.2f}")
print(f"{len(spawns.events)} pedestrian starts in {spawns.duration_s:g} s")

## The Driver Model

The safety requirements are the limits a drive must stay within: the driver aims to keep them, and the evaluation checks them.

The driver model adds how the car drives: how far ahead it notices another road user in clear conditions, and the braking it plans with. Both change with the actual conditions. A visibility loss shrinks the range, and a wet road lowers the braking the driver assumes.

In [ ]:
safety = DrivingSafetyReq(min_gap_m=2.0, max_deceleration_mps2=6.0, max_acceleration_mps2=3.0, speed_tolerance_kmh=3.0)
driver = DriverModel(safety=safety, detection_range_m=60.0, comfortable_deceleration_mps2=3.0)

wet, fog = spawns.condition_severity(hazards.wet_road), spawns.condition_severity(hazards.low_visibility)
print(f"sight in this run's rain or fog:  {driver.detection_range(fog):.0f} m of {driver.detection_range_m:g} m")
print(f"sight when also facing the sun:   {driver.detection_range(min(1.0, fog + spawns.sun.strength)):.0f} m")
print(f"braking planned on this wet road: {driver.planned_deceleration(wet):.1f} of {driver.comfortable_deceleration_mps2:g} m/s2")

## Scenario Files

A scenario is one file holding the road graph, the rolled hazards, the route and the driver. It is everything a run needs, so it is the only thing to copy to the machine that runs CARLA.

In [ ]:
for mode, route in routes.items():
    scenario = Driver.scenario(wrg, spawns, route, driver=driver, name=f"town10-seed{SEED}-{mode}")
    path = OUTPUT / f"{scenario.name}.json.gz"
    scenario.save(path)
    print(f"{path}  {route.estimated_time_s:5.1f} s planned, {path.stat().st_size / 1024:.0f} KB")

## Drive Them in CARLA

One command drives a scenario on a running CARLA 0.9.16 server and writes its telemetry beside it:

```bash
python -m pyotter drive runs/town10-seed1227-naive.json.gz
```

`use-pyotter/carla/README.md` has the launch scripts that start a server, drive, and stop it, for CARC Discovery, for macOS with the Wine wrapper, and for Docker. On the Mac, from the repo root:

```bash
use-pyotter/carla/wine/server start
use-pyotter/carla/wine/pyotter drive use-pyotter/runs/town10-seed1227-*.json.gz
use-pyotter/carla/wine/server stop
```

The rest of this notebook reads the telemetry files those runs leave in `runs/`.

In [ ]:
runs = {}
for mode in routes:
    path = OUTPUT / f"town10-seed{SEED}-{mode}-telemetry.json.gz"
    if path.exists():
        runs[mode] = DrivingScenarioTelemetry.load(path)
        run = runs[mode]
        print(f"{mode:11} {run.outcome}, {run.duration_s:5.1f} s, {run.distance_m:.0f} m, {len(run)} samples, "
              f"{len(run.events_of('hazard_started'))} pedestrians started, {len(run.events_of('collision'))} collisions")
    else:
        print(f"{mode:11} no telemetry yet: drive {path.name.replace('-telemetry', '')} first")

## Telemetry

Telemetry is a set of signals sampled once per tick, with a list of events. `gap_m` is the distance from the car's body to the nearest other road user.

In [ ]:
if runs:
    run = next(iter(runs.values()))
    print("signals:", ", ".join(run.signals))
    print("conditions:", {name: round(value, 2) for name, value in run.conditions.items()})
    print("first events:")
    for event in run.events[:4]:
        print(f"  {event.time_s:6.2f} s  {event.kind}  {event.detail}")

In [ ]:
if runs:
    figure = Visualize.telemetry(runs.values())
    figure.savefig(OUTPUT / "town10-telemetry.png", bbox_inches="tight")

## Evaluate

Each safety requirement is a signal temporal logic (STL) formula of the form `always (condition)`. Its **margin** is the formula's robustness, in the signal's own unit: positive is how much room was left at the closest moment, and negative is how far past the limit the run went. The percentage divides the margin by the limit, so requirements in different units can be compared.

The two acceleration limits count as breached only when exceeded for half a second without a break, because acceleration is measured tick to tick and a single tick can spike.

In [ ]:
evaluations = {mode: Driver.evaluate(run) for mode, run in runs.items()}
for evaluation in evaluations.values():
    print(evaluation.summary())
    print()

The requirement closest to its limit is the near miss to look at, whether or not anything was breached.

In [ ]:
for mode, evaluation in evaluations.items():
    closest = evaluation.closest
    print(f"{mode:11} {'kept all' if evaluation.satisfied else 'breached':9} closest: {closest.name}, margin {closest.robustness:+.2f} {closest.unit} "
          f"at {closest.worst_time_s:.1f} s")

The same telemetry can be checked against other requirements, without driving again.

In [ ]:
strict = DrivingSafetyReq(min_gap_m=6.0, max_deceleration_mps2=3.0)
for mode, run in runs.items():
    result = Driver.evaluate(run, strict)
    print(f"{mode:11} breached under the stricter limits: {[r.name for r in result.breached] or 'nothing'}")